In [6]:
from pathlib import Path

# Force project root to be parent of Backend
BASE_DIR = Path.cwd()

if BASE_DIR.name.lower() == "backend":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "Datasets"
ZIP_DIR = DATA_DIR / "13F_zip"

print("Project root:", BASE_DIR)

Project root: c:\Users\tyiho\DSE3101 Project\dse3101investmentproject


In [15]:
import zipfile
import pandas as pd
import os
from datetime import datetime

# ==========================================================
# 1️⃣  PATHS
# ==========================================================

zip_path = ZIP_DIR / "2013q2_form13f.zip"
extract_path = "temp_extracted"

# ==========================================================
# 2️⃣  EXTRACT ZIP
# ==========================================================

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# ==========================================================
# 3️⃣  LOAD TSV FILES
# ==========================================================

submission = pd.read_csv(os.path.join(extract_path, "SUBMISSION.tsv"), sep="\t")
coverpage = pd.read_csv(os.path.join(extract_path, "COVERPAGE.tsv"), sep="\t")
summarypage = pd.read_csv(os.path.join(extract_path, "SUMMARYPAGE.tsv"), sep="\t")
infotable = pd.read_csv(os.path.join(extract_path, "INFOTABLE.tsv"), sep="\t")

In [ ]:
# ==========================================================
# 4️⃣  FILTER SUBMISSION (13F-HR + 13F-HR/A only)
# ==========================================================

submission = submission[
    submission["SUBMISSIONTYPE"].isin(["13F-HR", "13F-HR/A"])
].copy()

# Convert date columns to datetime
submission["PERIODOFREPORT"] = pd.to_datetime(submission["PERIODOFREPORT"],format="%d-%b-%Y",errors="coerce")
submission["FILING_DATE"] = pd.to_datetime(submission["FILING_DATE"], format="%d-%b-%Y",errors="coerce"
)
# ==========================================================
# 5️⃣  JOIN COVERPAGE (for manager info + amendment flag)
# ==========================================================

submission = submission.merge(
    coverpage[[
        "ACCESSION_NUMBER",
        "FILINGMANAGER_NAME",
        "ISAMENDMENT"
    ]],
    on="ACCESSION_NUMBER",
    how="left"
)

# ==========================================================
# 6️⃣  AMENDMENT DEDUPLICATION
# Keep latest ACCESSION_NUMBER and latest FILING_DATE per (CIK, PERIODOFREPORT)
# ==========================================================

submission = submission.sort_values(
    ["CIK", "PERIODOFREPORT", "FILING_DATE", "ACCESSION_NUMBER"]
)

submission = (
    submission
    .groupby(["CIK", "PERIODOFREPORT"], as_index=False)
    .tail(1)
)

# ==========================================================
# 7️⃣  JOIN SUMMARYPAGE
# ==========================================================

submission = submission.merge(
    summarypage[[
        "ACCESSION_NUMBER",
        "TABLEVALUETOTAL",
        "TABLEENTRYTOTAL",
        "ISCONFIDENTIALOMITTED"
    ]],
    on="ACCESSION_NUMBER",
    how="left"
)



In [ ]:
# ==========================================================
# 8️⃣  JOIN INFOTABLE
# ==========================================================

df = infotable.merge(
    submission,
    on="ACCESSION_NUMBER",
    how="inner"
)

# ==========================================================
# 9️⃣  INFOTABLE FILTERS (Equity-only clean dataset)
# ==========================================================

# Keep shares only
df = df[df["SSHPRNAMTTYPE"] == "SH"]

# Remove options
df = df[df["PUTCALL"].isna()]

# Keep common stock only
df = df[df["TITLEOFCLASS"] == "COM"]

# Keep SOLE discretion only
df = df[df["INVESTMENTDISCRETION"] == "SOLE"]

# Drop invalid CUSIPs
df = df[df["CUSIP"].notna()]
df = df[df["CUSIP"] != "000000000"]

# ==========================================================
# 🔟  HANDLE VALUE UNIT BREAK (Pre-2023 multiply by 1000)
# ==========================================================

cutoff_date = pd.Timestamp("2023-01-03")

# Scale INFOTABLE VALUE
df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce")
df.loc[df["FILING_DATE"] < cutoff_date, "VALUE"] *= 1000

# Scale TABLEVALUETOTAL
df["TABLEVALUETOTAL"] = pd.to_numeric(df["TABLEVALUETOTAL"], errors="coerce")
df.loc[df["FILING_DATE"] < cutoff_date, "TABLEVALUETOTAL"] *= 1000

# ==========================================================
# 1️⃣1️⃣  COMPUTE PORTFOLIO WEIGHTS
# ==========================================================

df["weight"] = df["VALUE"] / df["TABLEVALUETOTAL"]

# ==========================================================
# 1️⃣2️⃣  FINAL COLUMN SELECTION
# ==========================================================

clean_df = df[[
    "CIK",
    "FILINGMANAGER_NAME",
    "PERIODOFREPORT",
    "FILING_DATE",
    "SUBMISSIONTYPE",
    "ISAMENDMENT",
    "TABLEVALUETOTAL",
    "TABLEENTRYTOTAL",
    "ISCONFIDENTIALOMITTED",
    "NAMEOFISSUER",
    "CUSIP",
    "TITLEOFCLASS",
    "VALUE",
    "SSHPRNAMT",
    "SSHPRNAMTTYPE",
    "INVESTMENTDISCRETION",
    "weight"
]].copy()

# ==========================================================
# 1️⃣3️⃣  SAVE CLEAN DATA
# ==========================================================

output_path = "clean_single_13f.csv"
clean_df.to_csv(output_path, index=False)

print("✅ Clean dataset saved to:", output_path)
print("Rows:", len(clean_df))

clean_df.head()

✅ Clean dataset saved to: clean_single_13f.csv
Rows: 11102


,CIK,FILINGMANAGER_NAME,PERIODOFREPORT,FILING_DATE,SUBMISSIONTYPE,ISAMENDMENT,TABLEVALUETOTAL,TABLEENTRYTOTAL,ISCONFIDENTIALOMITTED,NAMEOFISSUER,CUSIP,TITLEOFCLASS,VALUE,SSHPRNAMT,SSHPRNAMTTYPE,INVESTMENTDISCRETION,weight
0,1487438,"DONALDSON CAPITAL MANAGEMENT, LLC",2010-03-31,2013-06-27,13F-HR,NaN,225586000,85,N,***BP P L C SPONSORED ADR (FRM,055622104,COM,392000,6864,SH,SOLE,0.001738
1,1487438,"DONALDSON CAPITAL MANAGEMENT, LLC",2010-03-31,2013-06-27,13F-HR,NaN,225586000,85,N,3M COMPANY,88579Y101,COM,440000,5267,SH,SOLE,0.001950
2,1487438,"DONALDSON CAPITAL MANAGEMENT, LLC",2010-03-31,2013-06-27,13F-HR,NaN,225586000,85,N,ABBOTT LABORATORIES,002824100,COM,7940000,150715,SH,SOLE,0.035197
3,1487438,"DONALDSON CAPITAL MANAGEMENT, LLC",2010-03-31,2013-06-27,13F-HR,NaN,225586000,85,N,AT&T INC,00206R102,COM,392000,15176,SH,SOLE,0.001738
4,1487438,"DONALDSON CAPITAL MANAGEMENT, LLC",2010-03-31,2013-06-27,13F-HR,NaN,225586000,85,N,AUTOMATIC DATA PROCESSING INC,053015103,COM,793000,17835,SH,SOLE,0.003515
